In [4]:
# --- Minimal, focused notebook: headline results + per-month 24h cycles ---
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------- Config ----------------
TZ = "Europe/Copenhagen"
SYSTEMS_CSV   = "data/El - Sandbyvägen 196, Dalby.csv"
HOUSEHOLD_CSV = "data/El - Sandbyvägen 158, Dalby.csv"
PROD_COL = "Produktion"
IMP_COL  = "El kWh"
N_HOMES  = 50

BATTERY_KWH   = 36.0
SOC0_FRAC     = 0.80
ETA_RT        = 0.90
N_EV          = 20
EV_LEND_KWH   = 10.0
OFFICE_HOURS  = range(8, 17)  # 08–16:59 (50% EV availability)
MAX_CHARGE_KW = 10.0
MAX_DISCH_KW  = 10.0

# ---------------- IO ----------------
def read_european_table(dataset: str, datum="Datum", tz=TZ) -> pd.DataFrame:
    df = pd.read_table(dataset, sep=";", decimal=",", index_col=datum, parse_dates=[datum], na_values=["-","—"])
    # Localize robustly: shift nonexistent DST forward; ambiguous -> NaT (we drop them in groups)
    df.index = df.index.tz_localize(tz, nonexistent="shift_forward", ambiguous="NaT")
    return df

systems  = read_european_table(SYSTEMS_CSV)
household = read_european_table(HOUSEHOLD_CSV)

# ---------------- Time normalize ----------------
def norm_idx(idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    i = pd.to_datetime(idx, errors="coerce")
    return i.tz_convert(TZ).tz_localize(None) if getattr(i, "tz", None) is not None else i

sys_ = systems.copy();   sys_.index = norm_idx(sys_.index)
hh_  = household.copy(); hh_.index  = norm_idx(hh_.index)

# ---------------- Base series ----------------
if PROD_COL not in sys_.columns or IMP_COL not in sys_.columns:
    raise KeyError(f"Expected '{PROD_COL}' and '{IMP_COL}' in systems columns: {list(sys_.columns)}")
net_e = (sys_[PROD_COL] - sys_[IMP_COL]).rename("net_kWh")  # + surplus, - deficit

# Build household net if available (if no prod col in HH, treat as 0)
hh_imp_col = next((c for c in hh_.columns if c.lower() in {IMP_COL.lower(), "el kwh"}), None)
hh_prod_col= next((c for c in hh_.columns if c.lower() in {PROD_COL.lower(), "produktion"}), None)
if hh_imp_col is None:
    raise KeyError(f"Household import column not found in: {list(hh_.columns)}")
net_hh = ((hh_[hh_prod_col] if hh_prod_col else 0) - hh_[hh_imp_col]).rename("net_household")

# ---------------- 50 households with √N residual aggregation ----------------
mask = net_hh.index.notna()
s = net_hh[mask]
month_key = s.index.to_period("M")
hour_key  = s.index.to_series().dt.hour.astype(int)
trend_s = s.groupby([month_key, hour_key]).transform("mean")  # month×hour mean
trend = pd.Series(np.nan, index=net_hh.index, name="hh_trend"); trend.loc[s.index] = trend_s
resid = (net_hh - trend).rename("hh_resid")
net_hh_agg = (N_HOMES * trend + np.sqrt(N_HOMES) * resid).rename("net_households_agg")

aligned = pd.concat([net_e.rename("net_sys"), net_hh_agg], axis=1, join="inner").dropna()
net_connected_random = (aligned["net_sys"] + aligned["net_households_agg"]).rename("net_kWh")

# ---------------- Simulator ----------------
def _dt_hours(idx: pd.DatetimeIndex) -> np.ndarray:
    dt = pd.Series(idx).diff().dt.total_seconds().div(3600.0).to_numpy()
    if len(dt) > 1 and np.isnan(dt[1]): dt[0] = np.nanmedian(dt[1:])
    return np.where(np.isnan(dt), np.nanmedian(dt), dt)

def _ev_capacity_schedule(idx: pd.DatetimeIndex, n_ev=N_EV, ev_kwh=EV_LEND_KWH) -> pd.Series:
    hrs = idx.hour
    cap = np.where(np.isin(hrs, list(OFFICE_HOURS)), 0.5 * n_ev * ev_kwh, 1.0 * n_ev * ev_kwh)
    return pd.Series(cap, index=idx, name="ev_cap_kWh")

def simulate_series(e_kwh: pd.Series,
                    base_cap_kwh: float,
                    var_cap_kwh: pd.Series | None,
                    soc0_frac: float = SOC0_FRAC,
                    eta_rt: float   = ETA_RT,
                    max_chg_kw: float | None = MAX_CHARGE_KW,
                    max_dis_kw: float | None = MAX_DISCH_KW) -> pd.Series:
    eta = np.sqrt(eta_rt)
    idx = e_kwh.index
    e   = e_kwh.to_numpy(float)
    var = np.zeros_like(e) if var_cap_kwh is None else var_cap_kwh.reindex(idx).to_numpy(float)
    dt  = _dt_hours(idx)
    residual = np.zeros_like(e)
    soc_now = soc0_frac * (base_cap_kwh + var[0])
    for i, x in enumerate(e):
        cap_now = base_cap_kwh + var[i]
        if soc_now > cap_now:         # capacity drop (e.g., EVs depart)
            residual[i] += (soc_now - cap_now) / eta
            soc_now = cap_now
        ecap_in  = (max_chg_kw * dt[i]) if max_chg_kw is not None else np.inf
        ecap_out = (max_dis_kw * dt[i]) if max_dis_kw is not None else np.inf
        if x >= 0:                     # charge
            room = cap_now - soc_now
            charge_in = min(x * eta, room, ecap_in)
            spill = x * eta - charge_in
            soc_now += charge_in
            residual[i] = spill / eta
        else:                          # discharge
            need = -x
            deliverable = min(soc_now * eta, need, ecap_out)
            soc_now -= deliverable / eta
            residual[i] = -(need - deliverable)
    return pd.Series(residual, index=idx, name="net_after_kWh")

def import_kwh(s: pd.Series) -> float:
    return float(-s[s < 0].sum())

# ---------------- Scenarios: WITHOUT households (systems only) ----------------
ev_sched_sys = _ev_capacity_schedule(net_e.index)
net_sys_batt = simulate_series(net_e, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)
net_sys_vtx  = simulate_series(net_e, base_cap_kwh=0.0,         var_cap_kwh=ev_sched_sys)
net_sys_both = simulate_series(net_e, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sched_sys)

imp0_sys = import_kwh(net_e); imp_b_sys = import_kwh(net_sys_batt); imp_v_sys = import_kwh(net_sys_vtx); imp_bt_sys = import_kwh(net_sys_both)
print("\n[Without households]")
print(f"  Import ↓ battery-only: {imp0_sys-imp_b_sys:,.1f} kWh  ({(1-imp_b_sys/imp0_sys)*100:.1f}%)")
print(f"  Import ↓ VTX-only   : {imp0_sys-imp_v_sys:,.1f} kWh  ({(1-imp_v_sys/imp0_sys)*100:.1f}%)")
print(f"  Import ↓ BOTH       : {imp0_sys-imp_bt_sys:,.1f} kWh  ({(1-imp_bt_sys/imp0_sys)*100:.1f}%)")

# ---------------- Scenarios: WITH households (50, √N residual) ----------------
ev_sched_hh = _ev_capacity_schedule(net_connected_random.index)
net_hh_batt = simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)
net_hh_vtx  = simulate_series(net_connected_random, base_cap_kwh=0.0,         var_cap_kwh=ev_sched_hh)
net_hh_both = simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sched_hh)

imp0_hh = import_kwh(net_connected_random); imp_b_hh = import_kwh(net_hh_batt); imp_v_hh = import_kwh(net_hh_vtx); imp_bt_hh = import_kwh(net_hh_both)
print("\n[With households]")
print(f"  Imports (orig)      : {imp0_hh:,.1f} kWh")
print(f"  Imports after battery-only : {imp_b_hh:,.1f} kWh   (↓ {(1-imp_b_hh/imp0_hh)*100:.1f}%)")
print(f"  Imports after VTX-only     : {imp_v_hh:,.1f} kWh   (↓ {(1-imp_v_hh/imp0_hh)*100:.1f}%)")
print(f"  Imports after both         : {imp_bt_hh:,.1f} kWh   (↓ {(1-imp_bt_hh/imp0_hh)*100:.1f}%)")

# ---------------- Monthly 24h cycle grids ----------------
def month_grid(series: pd.Series, title: str, line_name: str):
    def hod_mean(mask):
        dh = series[mask].groupby([series[mask].index.normalize(), series[mask].index.hour]).sum().unstack(fill_value=0.0)
        hrs  = np.arange(24); xlbl = [f"{h:02d}:00" for h in hrs]
        mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        return xlbl, mean_
    names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    fig = make_subplots(rows=3, cols=4, subplot_titles=names, vertical_spacing=0.12, horizontal_spacing=0.06)
    for m in range(1, 13):
        mask = series.index.month == m
        if not mask.any(): continue
        xlbl, mean_ = hod_mean(mask); r = (m-1)//4 + 1; c = (m-1)%4 + 1
        fig.add_bar(x=xlbl, y=mean_, name="Mean (bar)" if m==1 else None,
                    marker_opacity=0.35, showlegend=(m==1), row=r, col=c)
        fig.add_scatter(x=xlbl, y=mean_, mode="lines", name=line_name if m==1 else None,
                        showlegend=(m==1), row=r, col=c)
        fig.update_xaxes(title_text="Hour", type="category", row=r, col=c)
        fig.update_yaxes(title_text="kWh/h", row=r, col=c)
    fig.update_layout(title=title, hovermode="x unified", height=900, barmode="overlay")
    fig.show()

# WITHOUT households (4 grids)
month_grid(net_e,          "Monthly 24h cycle — WITHOUT households: Original",     "Original mean")
month_grid(net_sys_batt,   "Monthly 24h cycle — WITHOUT households: Battery-only", "Battery-only")
month_grid(net_sys_vtx,    "Monthly 24h cycle — WITHOUT households: VTX-only",     "VTX-only")
month_grid(net_sys_both,   "Monthly 24h cycle — WITHOUT households: Battery+VTX",  "Battery+VTX")

# WITH households (4 grids)
month_grid(net_connected_random, "Monthly 24h cycle — WITH households: Original",     "Original (with HH)")
month_grid(net_hh_batt,          "Monthly 24h cycle — WITH households: Battery-only", "Battery-only")
month_grid(net_hh_vtx,           "Monthly 24h cycle — WITH households: VTX-only",     "VTX-only")
month_grid(net_hh_both,          "Monthly 24h cycle — WITH households: Battery+VTX",  "Battery+VTX")



[Without households]
  Import ↓ battery-only: 6,131.8 kWh  (8.7%)
  Import ↓ VTX-only   : 10,430.6 kWh  (14.7%)
  Import ↓ BOTH       : 10,754.0 kWh  (15.2%)

[With households]
  Imports (orig)      : 494,610.3 kWh
  Imports after battery-only : 492,934.9 kWh   (↓ 0.3%)
  Imports after VTX-only     : 492,626.1 kWh   (↓ 0.4%)
  Imports after both         : 492,598.8 kWh   (↓ 0.4%)


In [2]:
# Per-month small multiples: Systems vs Village (√N) in one figure
# Uses existing: pd, np, go, TZ, systems, household (and net_connected_random/net_sys if already defined)

from plotly.subplots import make_subplots

# --- Build only if missing (keeps your session clean) ---
if 'net_sys' not in globals():
    t = pd.to_datetime(systems.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    sys_ = systems.copy(); sys_.index = t
    net_sys = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys")

if 'net_connected_random' not in globals():
    # Minimal √N aggregation if not precomputed
    t = pd.to_datetime(household.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    hh_ = household.copy(); hh_.index = t
    hh_imp = next((c for c in hh_.columns if c.lower() in {"el kwh","el kwh "}), None)
    hh_prod = next((c for c in hh_.columns if "produktion" in c.lower() or "pv" in c.lower() or "solar" in c.lower()), None)
    net_hh = ((hh_[hh_prod] if hh_prod else 0) - hh_[hh_imp]).rename("net_household")

    N = 50
    s = net_hh[net_hh.index.notna()]
    trend_s = s.groupby([s.index.to_period("M"), s.index.to_series().dt.hour]).transform("mean")
    trend = pd.Series(np.nan, index=net_hh.index); trend.loc[s.index] = trend_s
    resid = net_hh - trend
    net_hh_agg = (N * trend + np.sqrt(N) * resid).rename("net_households_agg")

    aligned = pd.concat([net_sys.rename("net_sys"), net_hh_agg], axis=1, join="inner").dropna()
    net_connected_random = (aligned["net_sys"] + aligned["net_households_agg"]).rename("net_connected_random")

# --- Helpers ---
def _month_stats(series, m: int):
    mask = series.index.month == m
    if not mask.any():
        return None
    dh = series[mask].groupby([series[mask].index.normalize(),
                               series[mask].index.hour]).sum().unstack(fill_value=0.0)
    hrs = np.arange(24)
    xlbl = [f"{h:02d}:00" for h in hrs]
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return xlbl, mean_, vmin, vmax

# --- Single compact figure ---
names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig = make_subplots(rows=3, cols=4, subplot_titles=names, vertical_spacing=0.12, horizontal_spacing=0.06)

for m in range(1, 13):
    s_stats = _month_stats(net_sys, m)
    v_stats = _month_stats(net_connected_random, m)
    if s_stats is None or v_stats is None:
        continue
    xlbl, s_mean, s_min, s_max = s_stats
    _,    v_mean, v_min, v_max = v_stats
    r = (m-1)//4 + 1; c = (m-1)%4 + 1

    # Systems mean (bars, background)
    fig.add_bar(x=xlbl, y=s_mean,
                name="Systems mean (bar)" if m==1 else None,
                marker_opacity=0.35, showlegend=(m==1), row=r, col=c)

    # Systems min–max band (blue)
    fig.add_scatter(x=xlbl, y=s_max, mode="lines", line=dict(width=0), showlegend=False, row=r, col=c)
    fig.add_scatter(x=xlbl, y=s_min, mode="lines", fill="tonexty",
                    fillcolor="rgba(80,120,200,0.30)", line=dict(width=0),
                    name="Systems min–max" if m==1 else None, showlegend=(m==1),
                    row=r, col=c)

    # Village (50 homes, √N) min–max band (red)
    fig.add_scatter(x=xlbl, y=v_max, mode="lines", line=dict(width=0), showlegend=False, row=r, col=c)
    fig.add_scatter(x=xlbl, y=v_min, mode="lines", fill="tonexty",
                    fillcolor="rgba(200,80,80,0.25)", line=dict(width=0),
                    name="Village min–max (√N)" if m==1 else None, showlegend=(m==1),
                    row=r, col=c)

    # Village mean (line, red)
    fig.add_scatter(x=xlbl, y=v_mean, mode="lines",
                    name="Village mean" if m==1 else None, showlegend=(m==1),
                    row=r, col=c)

    fig.update_xaxes(title_text="Hour", type="category", row=r, col=c)
    fig.update_yaxes(title_text="kWh/h", row=r, col=c)

fig.update_layout(
    title="Per-month 24 h net — Systems (bars + min–max) vs Village (√N min–max + mean)",
    hovermode="x unified",
    height=900,
    barmode="overlay"
)
fig.show()


In [3]:
# === Unified per-month small multiples with fixed colors & integer hours, 4 scenarios ===
# Requires: pd, np, go, TZ; and preferably `net_sys`, `net_connected_random`,
# `simulate_series`, `_ev_capacity_schedule`, BATTERY_KWH, N_EV, EV_LEND_KWH.
# If some are missing, lightweight fallbacks are used.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------- Colors (fixed across all months) ----------------
SYS_BAR_COLOR   = "rgba(33,150,243,0.60)"   # blue bars (systems mean)
SYS_SPAN_FILL   = "rgba(33,150,243,0.25)"   # blue band (systems min–max)
VIL_LINE_COLOR  = "rgb(255,193,7)"          # yellow line (village mean)
VIL_SPAN_FILL   = "rgba(200,80,80,0.25)"    # red band (village min–max)

# ---------------- Fallbacks if helpers/series aren't present ----------------
def _dt_hours(idx: pd.DatetimeIndex) -> np.ndarray:
    dt = pd.Series(idx).diff().dt.total_seconds().div(3600.0).to_numpy()
    if len(dt) > 1 and np.isnan(dt[1]): dt[0] = np.nanmedian(dt[1:])
    return np.where(np.isnan(dt), np.nanmedian(dt), dt)

if 'simulate_series' not in globals():
    # Minimal simulator (same logic you've used)
    ETA_RT        = globals().get('ETA_RT', 0.90)
    MAX_CHARGE_KW = globals().get('MAX_CHARGE_KW', 10.0)
    MAX_DISCH_KW  = globals().get('MAX_DISCH_KW', 10.0)
    SOC0_FRAC     = globals().get('SOC0_FRAC', 0.80)
    def simulate_series(e_kwh: pd.Series, base_cap_kwh: float, var_cap_kwh: pd.Series | None,
                        soc0_frac: float = SOC0_FRAC, eta_rt: float = ETA_RT,
                        max_chg_kw: float | None = MAX_CHARGE_KW, max_dis_kw: float | None = MAX_DISCH_KW) -> pd.Series:
        eta = np.sqrt(eta_rt)
        idx = e_kwh.index
        e   = e_kwh.to_numpy(float)
        var = np.zeros_like(e) if var_cap_kwh is None else var_cap_kwh.reindex(idx).to_numpy(float)
        dt  = _dt_hours(idx)
        residual = np.zeros_like(e)
        soc_now = soc0_frac * (base_cap_kwh + var[0])
        for i, x in enumerate(e):
            cap_now = base_cap_kwh + var[i]
            if soc_now > cap_now:
                residual[i] += (soc_now - cap_now) / eta
                soc_now = cap_now
            ecap_in  = (max_chg_kw * dt[i]) if max_chg_kw is not None else np.inf
            ecap_out = (max_dis_kw * dt[i]) if max_dis_kw is not None else np.inf
            if x >= 0:
                room = cap_now - soc_now
                charge_in = min(x * eta, room, ecap_in)
                spill = x * eta - charge_in
                soc_now += charge_in
                residual[i] = spill / eta
            else:
                need = -x
                deliverable = min(soc_now * eta, need, ecap_out)
                soc_now -= deliverable / eta
                residual[i] = -(need - deliverable)
        return pd.Series(residual, index=idx, name="net_after_kWh")

if '_ev_capacity_schedule' not in globals():
    OFFICE_HOURS = globals().get('OFFICE_HOURS', range(8,17))
    N_EV         = globals().get('N_EV', 20)
    EV_LEND_KWH  = globals().get('EV_LEND_KWH', 10.0)
    def _ev_capacity_schedule(idx: pd.DatetimeIndex, n_ev=N_EV, ev_kwh=EV_LEND_KWH) -> pd.Series:
        hrs = idx.hour
        cap = np.where(np.isin(hrs, list(OFFICE_HOURS)), 0.5 * n_ev * ev_kwh, 1.0 * n_ev * ev_kwh)
        return pd.Series(cap, index=idx, name="ev_cap_kWh")

# Build `net_sys` if missing
if 'net_sys' not in globals():
    t = pd.to_datetime(systems.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    sys_ = systems.copy(); sys_.index = t
    net_sys = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys")

# Build `net_connected_random` if missing (simple N×mean + √N×resid model)
if 'net_connected_random' not in globals():
    t = pd.to_datetime(household.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    hh_ = household.copy(); hh_.index = t
    imp_col = next((c for c in hh_.columns if c.lower().strip() in {"el kwh","elkwh","el kwh"}), None)
    prod_col= next((c for c in hh_.columns if any(k in c.lower() for k in ["produktion","pv","solar","export"])), None)
    net_hh = ((hh_[prod_col] if prod_col else 0) - hh_[imp_col]).rename("net_household")
    N = 50
    s = net_hh[net_hh.index.notna()]
    trend_s = s.groupby([s.index.to_period("M"), s.index.to_series().dt.hour]).transform("mean")
    trend = pd.Series(np.nan, index=net_hh.index); trend.loc[s.index] = trend_s
    resid = net_hh - trend
    net_hh_agg = (N * trend + np.sqrt(N) * resid).rename("net_households_agg")
    aligned = pd.concat([net_sys.rename("net_sys"), net_hh_agg], axis=1, join="inner").dropna()
    net_connected_random = (aligned["net_sys"] + aligned["net_households_agg"]).rename("net_connected_random")

# Battery size if missing
BATTERY_KWH = globals().get('BATTERY_KWH', 36.0)

# ---------------- Per-month stats helper ----------------
def _month_stats(series: pd.Series, month: int):
    mask = series.index.month == month
    if not mask.any(): return None
    dh = series[mask].groupby([series[mask].index.normalize(),
                               series[mask].index.hour]).sum().unstack(fill_value=0.0)
    hrs = np.arange(24)
    x = hrs  # integers 0..23
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return x, mean_, vmin, vmax

# ---------------- One figure builder per scenario ----------------
def plot_month_grid(sys_series: pd.Series, vil_series: pd.Series, title: str):
    names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    fig = make_subplots(rows=3, cols=4, subplot_titles=names, vertical_spacing=0.12, horizontal_spacing=0.06)
    for m in range(1, 13):
        s_stats = _month_stats(sys_series, m)
        v_stats = _month_stats(vil_series, m)
        if s_stats is None or v_stats is None: continue
        x, s_mean, s_min, s_max = s_stats
        _, v_mean, v_min, v_max = v_stats
        r = (m-1)//4 + 1; c = (m-1)%4 + 1

        # Systems mean bars (BLUE)
        fig.add_bar(x=x, y=s_mean, name="Systems mean", marker_color=SYS_BAR_COLOR,
                    showlegend=(m==1), row=r, col=c)

        # Systems min–max band (BLUE)
        fig.add_scatter(x=x, y=s_max, mode="lines", line=dict(width=0),
                        showlegend=False, row=r, col=c)
        fig.add_scatter(x=x, y=s_min, mode="lines", fill="tonexty",
                        fillcolor=SYS_SPAN_FILL, line=dict(width=0),
                        name="Systems min–max", showlegend=(m==1), row=r, col=c)

        # Village min–max band (RED)
        fig.add_scatter(x=x, y=v_max, mode="lines", line=dict(width=0),
                        showlegend=False, row=r, col=c)
        fig.add_scatter(x=x, y=v_min, mode="lines", fill="tonexty",
                        fillcolor=VIL_SPAN_FILL, line=dict(width=0),
                        name="Village min–max (√N)", showlegend=(m==1), row=r, col=c)

        # Village mean line (YELLOW)
        fig.add_scatter(x=x, y=v_mean, mode="lines",
                        line=dict(color=VIL_LINE_COLOR, width=2),
                        name="Village mean", showlegend=(m==1), row=r, col=c)

        fig.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=r, col=c)
        fig.update_yaxes(title_text="kWh/h", row=r, col=c)

    fig.update_layout(
        title=title, hovermode="x unified", height=900, barmode="overlay",
        legend=dict(orientation="h", x=0, xanchor="left", y=1.08)
    )
    fig.show()

# ---------------- Build scenarios ----------------
ev_sched_sys = _ev_capacity_schedule(net_sys.index)
ev_sched_vil = _ev_capacity_schedule(net_connected_random.index)

# Original (no storage)
sys_orig = net_sys
vil_orig = net_connected_random

# Battery-only
sys_batt = simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)
vil_batt = simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)

# VTX-only
sys_vtx  = simulate_series(net_sys, base_cap_kwh=0.0, var_cap_kwh=ev_sched_sys)
vil_vtx  = simulate_series(net_connected_random, base_cap_kwh=0.0, var_cap_kwh=ev_sched_vil)

# Battery + VTX
sys_both = simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sched_sys)
vil_both = simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sched_vil)

# ---------------- Plot the 4 scenario figures ----------------
plot_month_grid(sys_orig, vil_orig, "Monthly 24 h — Original (Systems vs Village √N)")
plot_month_grid(sys_batt, vil_batt, f"Monthly 24 h — Battery-only ({BATTERY_KWH:.0f} kWh) (Systems vs Village √N)")
plot_month_grid(sys_vtx,  vil_vtx,  "Monthly 24 h — VTX-only (Systems vs Village √N)")
plot_month_grid(sys_both, vil_both, f"Monthly 24 h — Battery + VTX ({BATTERY_KWH:.0f} kWh) (Systems vs Village √N)")


In [4]:
# === Year-averaged daily cycles, side-by-side (4 scenarios, shared Y) ===
# Uses existing: pd, np, go, TZ, systems, household,
# and if available: net_sys, net_connected_random, simulate_series, _ev_capacity_schedule, BATTERY_KWH, N_EV, EV_LEND_KWH.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- Colors ----------
SYS_BAR_COLOR = "rgba(33,150,243,0.60)"   # blue bars
SYS_SPAN_FILL = "rgba(33,150,243,0.25)"   # blue band (systems min–max)
VIL_LINE_COLOR= "rgb(255,193,7)"          # yellow line

# ---------- Fallbacks if needed (kept tight) ----------
def _dt_hours(idx):
    dt = pd.Series(idx).diff().dt.total_seconds().div(3600.0).to_numpy()
    if len(dt) > 1 and np.isnan(dt[1]): dt[0] = np.nanmedian(dt[1:])
    return np.where(np.isnan(dt), np.nanmedian(dt), dt)

if 'simulate_series' not in globals():
    ETA_RT = globals().get('ETA_RT', 0.90)
    MAX_CHARGE_KW = globals().get('MAX_CHARGE_KW', 10.0)
    MAX_DISCH_KW  = globals().get('MAX_DISCH_KW', 10.0)
    SOC0_FRAC     = globals().get('SOC0_FRAC', 0.80)
    def simulate_series(e_kwh, base_cap_kwh, var_cap_kwh,
                        soc0_frac=SOC0_FRAC, eta_rt=ETA_RT,
                        max_chg_kw=MAX_CHARGE_KW, max_dis_kw=MAX_DISCH_KW):
        eta = np.sqrt(eta_rt); idx = e_kwh.index
        e = e_kwh.to_numpy(float)
        var = np.zeros_like(e) if var_cap_kwh is None else var_cap_kwh.reindex(idx).to_numpy(float)
        dt = _dt_hours(idx); res = np.zeros_like(e); soc = soc0_frac * (base_cap_kwh + var[0])
        for i, x in enumerate(e):
            cap = base_cap_kwh + var[i]
            if soc > cap: res[i] += (soc-cap)/eta; soc = cap
            ecap_in  = (max_chg_kw*dt[i]) if max_chg_kw is not None else np.inf
            ecap_out = (max_dis_kw*dt[i]) if max_dis_kw is not None else np.inf
            if x >= 0:
                room = cap - soc
                charge = min(x*eta, room, ecap_in)
                spill = x*eta - charge
                soc += charge
                res[i] = spill/eta
            else:
                need = -x
                give = min(soc*eta, need, ecap_out)
                soc -= give/eta
                res[i] = -(need - give)
        return pd.Series(res, index=idx, name="net_after_kWh")

if '_ev_capacity_schedule' not in globals():
    OFFICE_HOURS = globals().get('OFFICE_HOURS', range(8,17))
    N_EV        = globals().get('N_EV', 20)
    EV_LEND_KWH = globals().get('EV_LEND_KWH', 10.0)
    def _ev_capacity_schedule(idx, n_ev=N_EV, ev_kwh=EV_LEND_KWH):
        hrs = idx.hour
        cap = np.where(np.isin(hrs, list(OFFICE_HOURS)), 0.5*n_ev*ev_kwh, 1.0*n_ev*ev_kwh)
        return pd.Series(cap, index=idx, name="ev_cap_kWh")

# Build net_sys if missing
if 'net_sys' not in globals():
    t = pd.to_datetime(systems.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    sys_ = systems.copy(); sys_.index = t
    net_sys = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys")

# Build net_connected_random if missing (√N aggregation)
if 'net_connected_random' not in globals():
    t = pd.to_datetime(household.index)
    t = t.tz_convert(TZ).tz_localize(None) if getattr(t, "tz", None) is not None else t
    hh_ = household.copy(); hh_.index = t
    imp_col = next((c for c in hh_.columns if c.lower().strip() in {"el kwh","elkwh"}), None)
    prod_col= next((c for c in hh_.columns if any(k in c.lower() for k in ["produktion","pv","solar","export"])), None)
    net_hh = ((hh_[prod_col] if prod_col else 0) - hh_[imp_col]).rename("net_household")
    N = 50
    s = net_hh[net_hh.index.notna()]
    trend_s = s.groupby([s.index.to_period("M"), s.index.to_series().dt.hour]).transform("mean")
    trend = pd.Series(np.nan, index=net_hh.index); trend.loc[s.index] = trend_s
    resid = net_hh - trend
    net_hh_agg = (N*trend + np.sqrt(N)*resid).rename("net_households_agg")
    aligned = pd.concat([net_sys.rename("net_sys"), net_hh_agg], axis=1, join="inner").dropna()
    net_connected_random = (aligned["net_sys"] + aligned["net_households_agg"]).rename("net_connected_random")

BATTERY_KWH = globals().get('BATTERY_KWH', 36.0)

# ---------- Hour-of-day stats over the whole year ----------
def hod_stats(series: pd.Series):
    # day×hour matrix; sum per clock hour to tolerate non-hourly cadence
    dh = series.groupby([series.index.normalize(), series.index.hour]).sum().unstack(fill_value=0.0)
    hrs = np.arange(24)
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return hrs, mean_, vmin, vmax

# ---------- Build scenarios (systems & village) ----------
ev_sys = _ev_capacity_schedule(net_sys.index)
ev_vil = _ev_capacity_schedule(net_connected_random.index)

scenarios = {
    "Original":      (net_sys,                                  net_connected_random),
    "Battery-only":  (simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None),
                      simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)),
    "VTX-only":      (simulate_series(net_sys, base_cap_kwh=0.0, var_cap_kwh=ev_sys),
                      simulate_series(net_connected_random, base_cap_kwh=0.0, var_cap_kwh=ev_vil)),
    "Battery + VTX": (simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sys),
                      simulate_series(net_connected_random, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_vil)),
}

# Precompute stats and global y-range
stats = {}
y_vals = []
for name, (sys_s, vil_s) in scenarios.items():
    x, s_mean, s_min, s_max = hod_stats(sys_s)
    _, v_mean, v_min, v_max = hod_stats(vil_s)
    stats[name] = (x, s_mean, s_min, s_max, v_mean)  # we plot village mean only
    y_vals.extend([s_min.min(), s_max.max(), v_mean.min(), v_mean.max()])
y_min, y_max = float(min(y_vals)), float(max(y_vals))

# ---------- Plot (1×4) ----------
fig = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)

for j, name in enumerate(scenarios.keys(), start=1):
    x, s_mean, s_min, s_max, v_mean = stats[name]

    # Systems mean bars
    fig.add_bar(x=x, y=s_mean, marker_color=SYS_BAR_COLOR,
                name="Systems mean" if j==1 else None, showlegend=(j==1), row=1, col=j)

    # Systems min–max band
    fig.add_scatter(x=x, y=s_max, mode="lines", line=dict(width=0),
                    showlegend=False, row=1, col=j)
    fig.add_scatter(x=x, y=s_min, mode="lines", fill="tonexty",
                    fillcolor=SYS_SPAN_FILL, line=dict(width=0),
                    name="Systems min–max" if j==1 else None, showlegend=(j==1), row=1, col=j)

    # Village mean line
    fig.add_scatter(x=x, y=v_mean, mode="lines",
                    line=dict(color=VIL_LINE_COLOR, width=2),
                    name="Village mean" if j==1 else None, showlegend=(j==1), row=1, col=j)

    fig.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig.update_yaxes(range=[y_min, y_max], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig.update_layout(
    title=f"Year-averaged daily cycles — Systems (bars+span) vs Village (mean) — Battery {BATTERY_KWH:.0f} kWh",
    hovermode="x unified",
    height=500,
    barmode="overlay",
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig.show()


In [5]:
# Fix √N aggregation (drop NaT before grouping), then plot 1×4 year-averaged daily cycles
# Uses: systems, household, pd, np, go, TZ, simulate_series, _ev_capacity_schedule, BATTERY_KWH

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_HOMES = 50
BATTERY_KWH = globals().get("BATTERY_KWH", 36.0)

# Colors
SYS_BAR_COLOR = "rgba(33,150,243,0.60)"   # systems mean
SYS_SPAN_FILL = "rgba(33,150,243,0.25)"   # systems min–max
VIL_LINE_COLOR= "rgb(255,193,7)"          # village mean
VIL_SPAN_FILL = "rgba(255,193,7,0.22)"    # village min–max

def _norm_idx(idx):
    i = pd.to_datetime(idx, errors="coerce")
    return i.tz_convert(TZ).tz_localize(None) if getattr(i, "tz", None) is not None else i

def hod_stats(series: pd.Series):
    s = series.dropna()
    dh = s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)
    hrs = np.arange(24)
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return hrs, mean_, vmin, vmax

# Systems net (local, NaT-safe)
sys_ = systems.copy(); sys_.index = _norm_idx(sys_.index)
net_sys = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys").dropna()

# Household net (local, NaT-safe)
hh_ = household.copy(); hh_.index = _norm_idx(hh_.index)
hh_imp = next((c for c in hh_.columns if c.lower().strip() in {"el kwh","elkwh"}), None)
hh_prod= next((c for c in hh_.columns if any(k in c.lower() for k in ["produktion","pv","solar","export"])), None)
if hh_imp is None:
    raise KeyError(f"Household import column not found in: {list(hh_.columns)}")
net_hh = ((hh_[hh_prod] if hh_prod else 0) - hh_[hh_imp]).rename("net_household").dropna()

# √N aggregation: build only on overlapping, valid timestamps
idx = net_sys.index.intersection(net_hh.index)            # align timelines
s = net_hh.reindex(idx).dropna()
# avoid NaT hour: s has no NaT now
trend = s.groupby([s.index.to_period("M"), s.index.to_series().dt.hour]).transform("mean")
resid  = s - trend
net_hh_agg = (N_HOMES * trend + np.sqrt(N_HOMES) * resid).rename("net_households_agg")

# Village series
net_sys_aligned = net_sys.reindex(idx).dropna()
village_base = (net_sys_aligned + net_hh_agg).rename("net_village_sqrtN")

# Sanity check: residual std scale ~ √N (dropna and compare)
hh_resid = (s - s.groupby(s.index.hour).transform("mean")).dropna()
vil_resid = (village_base - village_base.groupby(village_base.index.hour).transform("mean")).dropna()
if not hh_resid.empty and not vil_resid.empty:
    r = float(vil_resid.std() / hh_resid.std())
    print(f"[check] Residual std scale ≈ {r:.2f} (expect ~ {np.sqrt(N_HOMES):.2f})")

# Scenarios (systems & village)
ev_sys = _ev_capacity_schedule(net_sys_aligned.index)
ev_vil = _ev_capacity_schedule(village_base.index)

scenarios = {
    "Original":      (net_sys_aligned, village_base),
    "Battery-only":  (simulate_series(net_sys_aligned, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None),
                      simulate_series(village_base,     base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)),
    "VTX-only":      (simulate_series(net_sys_aligned, base_cap_kwh=0.0, var_cap_kwh=ev_sys),
                      simulate_series(village_base,     base_cap_kwh=0.0, var_cap_kwh=ev_vil)),
    "Battery + VTX": (simulate_series(net_sys_aligned, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sys),
                      simulate_series(village_base,     base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_vil)),
}

# Precompute stats + shared Y
stats = {}
y_vals = []
for name, (s_sys, s_vil) in scenarios.items():
    x, s_mean, s_min, s_max = hod_stats(s_sys)
    _, v_mean, v_min, v_max = hod_stats(s_vil)
    stats[name] = (x, s_mean, s_min, s_max, v_mean, v_min, v_max)
    y_vals += [s_min.min(), s_max.max(), v_min.min(), v_max.max(), v_mean.min(), v_mean.max()]
y_min, y_max = float(min(y_vals)), float(max(y_vals))

# Plot 1×4
fig = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, s_mean, s_min, s_max, v_mean, v_min, v_max = stats[name]
    fig.add_bar(x=x, y=s_mean, marker_color=SYS_BAR_COLOR,
                name="Systems mean" if j==1 else None, showlegend=(j==1), row=1, col=j)
    fig.add_scatter(x=x, y=s_max, mode="lines", line=dict(width=0), showlegend=False, row=1, col=j)
    fig.add_scatter(x=x, y=s_min, mode="lines", fill="tonexty", fillcolor=SYS_SPAN_FILL, line=dict(width=0),
                    name="Systems min–max" if j==1 else None, showlegend=(j==1), row=1, col=j)
    fig.add_scatter(x=x, y=v_max, mode="lines", line=dict(width=0), showlegend=False, row=1, col=j)
    fig.add_scatter(x=x, y=v_min, mode="lines", fill="tonexty", fillcolor=VIL_SPAN_FILL, line=dict(width=0),
                    name="Village min–max (√N)", showlegend=(j==1), row=1, col=j)
    fig.add_scatter(x=x, y=v_mean, mode="lines",
                    line=dict(color=VIL_LINE_COLOR, width=2),
                    name="Village mean", showlegend=(j==1), row=1, col=j)
    fig.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig.update_yaxes(range=[y_min, y_max], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig.update_layout(
    title=f"Year-averaged daily cycles — Systems (blue) vs Village (yellow, √{N_HOMES})",
    hovermode="x unified", height=520, barmode="overlay",
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig.show()


[check] Residual std scale ≈ 53.31 (expect ~ 7.07)


In [6]:
# Year-averaged daily cycles with correct √N (month-wise) aggregation for village, 4 scenarios side-by-side
# Uses existing: systems, household, pd, np, go, TZ, simulate_series, _ev_capacity_schedule, BATTERY_KWH

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_HOMES = 50
BATTERY_KWH = globals().get("BATTERY_KWH", 36.0)

# Fixed colors
SYS_BAR_COLOR = "rgba(33,150,243,0.60)"   # systems mean (blue)
SYS_SPAN_FILL = "rgba(33,150,243,0.25)"   # systems min–max (blue)
VIL_LINE_COLOR= "rgb(255,193,7)"          # village mean (yellow)
VIL_SPAN_FILL = "rgba(255,193,7,0.22)"    # village min–max (yellow)

def _norm_idx(idx):
    i = pd.to_datetime(idx, errors="coerce")
    return i.tz_convert(TZ).tz_localize(None) if getattr(i, "tz", None) is not None else i

def _hod_stats(series: pd.Series):
    s = series.dropna()
    dh = s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)
    hrs = np.arange(24)
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return hrs, mean_, vmin, vmax

# --- Build base series (NaT-safe) ---
sys_ = systems.copy();   sys_.index = _norm_idx(sys_.index)
net_sys = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys").dropna()

hh_  = household.copy(); hh_.index  = _norm_idx(hh_.index)
hh_imp = next((c for c in hh_.columns if c.lower().strip() in {"el kwh","elkwh"}), None)
hh_prod= next((c for c in hh_.columns if any(k in c.lower() for k in ["produktion","pv","solar","export"])), None)
if hh_imp is None:
    raise KeyError(f"Household import column not found in: {list(hh_.columns)}")
net_hh = ((hh_[hh_prod] if hh_prod else 0) - hh_[hh_imp]).rename("net_household").dropna()

# --- Month-wise √N aggregation for village, then stitch months ---
pieces = []
for m in range(1, 13):
    # common timestamps in this month
    mask_sys = net_sys.index.month == m
    mask_hh  = net_hh.index.month  == m
    if not mask_sys.any() or not mask_hh.any():
        continue
    idx_m = net_sys.index[mask_sys].intersection(net_hh.index[mask_hh])
    if idx_m.empty:
        continue

    s_hh = net_hh.reindex(idx_m).dropna()
    # trend = hour-of-day mean **within this month**
    hod_mean_m = s_hh.groupby(s_hh.index.hour).transform("mean")
    resid_m = s_hh - hod_mean_m

    net_hh_agg_m = (N_HOMES * hod_mean_m + np.sqrt(N_HOMES) * resid_m).rename("net_households_agg_m")
    net_sys_m = net_sys.reindex(idx_m).dropna()

    village_m = (net_sys_m + net_hh_agg_m).rename("net_village_sqrtN")
    pieces.append(village_m)

if not pieces:
    raise RuntimeError("No overlapping monthly data found to build village series.")
village_year_sqrtN = pd.concat(pieces).sort_index()

# --- Scenarios (systems & village) ---
ev_sys = _ev_capacity_schedule(net_sys.index)
ev_vil = _ev_capacity_schedule(village_year_sqrtN.index)

scenarios = {
    "Original":      (net_sys,                                    village_year_sqrtN),
    "Battery-only":  (simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None),
                      simulate_series(village_year_sqrtN, base_cap_kwh=BATTERY_KWH, var_cap_kwh=None)),
    "VTX-only":      (simulate_series(net_sys, base_cap_kwh=0.0, var_cap_kwh=ev_sys),
                      simulate_series(village_year_sqrtN, base_cap_kwh=0.0, var_cap_kwh=ev_vil)),
    "Battery + VTX": (simulate_series(net_sys, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_sys),
                      simulate_series(village_year_sqrtN, base_cap_kwh=BATTERY_KWH, var_cap_kwh=ev_vil)),
}

# Precompute year HOD stats + shared y-range
stats = {}
y_vals = []
for name, (s_sys, s_vil) in scenarios.items():
    x, s_mean, s_min, s_max = _hod_stats(s_sys)
    _, v_mean, v_min, v_max = _hod_stats(s_vil)
    stats[name] = (x, s_mean, s_min, s_max, v_mean, v_min, v_max)
    y_vals += [s_min.min(), s_max.max(), v_min.min(), v_max.max(), v_mean.min(), v_mean.max()]
y_min, y_max = float(min(y_vals)), float(max(y_vals))

# --- Plot: 1×4 panels ---
fig = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, s_mean, s_min, s_max, v_mean, v_min, v_max = stats[name]

    # Systems: mean bars + min–max band (blue)
    fig.add_bar(x=x, y=s_mean, marker_color=SYS_BAR_COLOR,
                name="Systems mean" if j==1 else None, showlegend=(j==1), row=1, col=j)
    fig.add_scatter(x=x, y=s_max, mode="lines", line=dict(width=0), showlegend=False, row=1, col=j)
    fig.add_scatter(x=x, y=s_min, mode="lines", fill="tonexty",
                    fillcolor=SYS_SPAN_FILL, line=dict(width=0),
                    name="Systems min–max" if j==1 else None, showlegend=(j==1), row=1, col=j)

    # Village: min–max band + mean line (yellow)
    fig.add_scatter(x=x, y=v_max, mode="lines", line=dict(width=0), showlegend=False, row=1, col=j)
    fig.add_scatter(x=x, y=v_min, mode="lines", fill="tonexty",
                    fillcolor=VIL_SPAN_FILL, line=dict(width=0),
                    name="Village min–max (√N)", showlegend=(j==1), row=1, col=j)
    fig.add_scatter(x=x, y=v_mean, mode="lines",
                    line=dict(color=VIL_LINE_COLOR, width=2),
                    name="Village mean", showlegend=(j==1), row=1, col=j)

    fig.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig.update_yaxes(range=[y_min, y_max], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig.update_layout(
    title=f"Year-averaged daily cycles (month-wise √{N_HOMES} aggregation) — Systems (blue) vs Village (yellow)",
    hovermode="x unified", height=520, barmode="overlay",
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig.show()


In [7]:
# === Separate figures: Systems-only vs Whole-village (year-averaged daily cycles) ===
# Requires you already built `scenarios = { "Original": (sys_series, village_series), ... }`.
# If not, run the previous scenario-building cell first.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Choose band mode ----
BAND_MODE = "percentile"    # 'global' | 'percentile' | 'within_month'
PCT_LOW, PCT_HIGH = 10, 90  # used only when BAND_MODE == 'percentile'

# ---- Fixed colors ----
SYS_BAR_COLOR = "rgba(33,150,243,0.60)"   # systems bars (blue)
SYS_SPAN_FILL = "rgba(33,150,243,0.25)"   # systems band (blue)
VIL_BAR_COLOR = "rgba(255,193,7,0.85)"    # village bars (yellow)
VIL_SPAN_FILL = "rgba(255,193,7,0.22)"    # village band (yellow)

def _day_hour_matrix(series: pd.Series) -> pd.DataFrame:
    s = series.dropna()
    # Aggregate per day × clock-hour to tolerate non-hourly cadence
    return s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)

def hod_mean_and_band(series: pd.Series, mode: str):
    """Return (hrs0_23, mean, band_low, band_high) per selected band mode."""
    hrs = np.arange(24)
    if mode == "global":
        dh = _day_hour_matrix(series)
        mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        low   = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        high  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        return hrs, mean_, low, high

    if mode == "percentile":
        dh = _day_hour_matrix(series)
        mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        low   = dh.quantile(PCT_LOW/100.0,  axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        high  = dh.quantile(PCT_HIGH/100.0, axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        return hrs, mean_, low, high

    if mode == "within_month":
        months = sorted(series.dropna().index.to_period("M").unique())
        per_m_mean, per_m_low, per_m_high = [], [], []
        for m in months:
            mask = (series.index.to_period("M") == m)
            if not mask.any():
                continue
            dh_m = _day_hour_matrix(series[mask])
            per_m_mean.append(dh_m.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
            per_m_low.append( dh_m.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
            per_m_high.append(dh_m.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
        if not per_m_mean:  # fallback
            return hod_mean_and_band(series, "global")
        mean_ = np.median(np.vstack(per_m_mean), axis=0)
        low   = np.median(np.vstack(per_m_low),  axis=0)
        high  = np.median(np.vstack(per_m_high), axis=0)
        return hrs, mean_, low, high

    raise ValueError("BAND_MODE must be 'global' | 'percentile' | 'within_month'")

if 'scenarios' not in globals():
    raise RuntimeError("Missing `scenarios`. Please run the scenario-building cell first.")

# ---- Precompute stats for all scenarios (systems & village) and shared Y ranges ----
sys_stats, vil_stats = {}, {}
sys_y_vals, vil_y_vals = [], []

for name, (sys_s, vil_s) in scenarios.items():
    x, s_mean, s_low, s_high = hod_mean_and_band(sys_s, BAND_MODE)
    _, v_mean, v_low, v_high = hod_mean_and_band(vil_s, BAND_MODE)
    sys_stats[name] = (x, s_mean, s_low, s_high)
    vil_stats[name] = (x, v_mean, v_low, v_high)
    sys_y_vals += [s_low.min(), s_high.max(), s_mean.min(), s_mean.max()]
    vil_y_vals += [v_low.min(), v_high.max(), v_mean.min(), v_mean.max()]

sys_ymin, sys_ymax = float(min(sys_y_vals)), float(max(sys_y_vals))
vil_ymin, vil_ymax = float(min(vil_y_vals)), float(max(vil_y_vals))

title_mode = {
    "global": "Global min–max (year-wide)",
    "percentile": f"P{PCT_LOW}–P{PCT_HIGH} band (robust)",
    "within_month": "Within-month median min–max (typical)"
}

# ---- Figure 1: Systems-only (blue) ----
fig_sys = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, mean_, low, high = sys_stats[name]
    fig_sys.add_bar(x=x, y=mean_, marker_color=SYS_BAR_COLOR,
                    name="Systems mean" if j==1 else None, showlegend=(j==1), row=1, col=j)
    fig_sys.add_scatter(x=x, y=high, mode="lines", line=dict(width=0),
                        showlegend=False, row=1, col=j)
    fig_sys.add_scatter(x=x, y=low, mode="lines", fill="tonexty",
                        fillcolor=SYS_SPAN_FILL, line=dict(width=0),
                        name=("Systems band" if j==1 else None), showlegend=(j==1), row=1, col=j)
    fig_sys.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig_sys.update_yaxes(range=[sys_ymin, sys_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_sys.update_layout(
    title=f"Year-averaged daily cycles — Systems only (blue) — band: {title_mode[BAND_MODE]}",
    hovermode="x unified", height=480, barmode="overlay",
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_sys.show()

# ---- Figure 2: Whole-village (yellow bars + yellow span) ----
fig_vil = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, mean_, low, high = vil_stats[name]
    fig_vil.add_bar(x=x, y=mean_, marker_color=VIL_BAR_COLOR,
                    name="Village mean" if j==1 else None, showlegend=(j==1), row=1, col=j)
    fig_vil.add_scatter(x=x, y=high, mode="lines", line=dict(width=0),
                        showlegend=False, row=1, col=j)
    fig_vil.add_scatter(x=x, y=low, mode="lines", fill="tonexty",
                        fillcolor=VIL_SPAN_FILL, line=dict(width=0),
                        name=("Village band" if j==1 else None), showlegend=(j==1), row=1, col=j)
    fig_vil.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig_vil.update_yaxes(range=[vil_ymin, vil_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_vil.update_layout(
    title=f"Year-averaged daily cycles — Whole village (yellow) — band: {title_mode[BAND_MODE]}",
    hovermode="x unified", height=480, barmode="overlay",
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_vil.show()


In [8]:
# High-contrast rendering for the 1×4 panels (Systems-only and Whole-village)
# Assumes `scenarios` already exists from the previous cell.
# Optional: set BAND_MODE in {"percentile","within_month","global"}; defaults to "percentile".

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Params ----
BAND_MODE   = globals().get("BAND_MODE", "percentile")
PCT_LOW     = globals().get("PCT_LOW", 10)
PCT_HIGH    = globals().get("PCT_HIGH", 90)

# ---- Colors & styles (higher contrast) ----
SYS_BAR_CLR     = "rgba(33,150,243,0.25)"   # lighter blue bars (more see-through)
SYS_BAR_EDGE    = "rgb(21,101,192)"         # darker blue outline
SYS_BAND_FILL   = "rgba(33,150,243,0.35)"   # blue band (slightly stronger)
SYS_BAND_EDGE   = "rgb(13,71,161)"          # dark blue band edge

VIL_BAR_CLR     = "rgba(255,193,7,0.25)"    # lighter yellow bars
VIL_BAR_EDGE    = "rgb(204,154,0)"          # darker yellow outline
VIL_BAND_FILL   = "rgba(255,193,7,0.40)"    # yellow band (slightly stronger)
VIL_BAND_EDGE   = "rgb(218,145,0)"          # darker yellow band edge

BARGAP = 0.35  # more air between columns

def _day_hour_matrix(series: pd.Series) -> pd.DataFrame:
    s = series.dropna()
    return s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)

def hod_mean_and_band(series: pd.Series, mode: str):
    hrs = np.arange(24)
    if mode == "global":
        dh = _day_hour_matrix(series)
        mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        low   = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        high  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        return hrs, mean_, low, high
    if mode == "percentile":
        dh = _day_hour_matrix(series)
        mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        low   = dh.quantile(PCT_LOW/100.0,  axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        high  = dh.quantile(PCT_HIGH/100.0, axis=0).reindex(hrs, fill_value=0.0).to_numpy()
        return hrs, mean_, low, high
    if mode == "within_month":
        months = sorted(series.dropna().index.to_period("M").unique())
        per_m_mean, per_m_low, per_m_high = [], [], []
        for m in months:
            mask = (series.index.to_period("M") == m)
            if not mask.any(): continue
            dh = _day_hour_matrix(series[mask])
            per_m_mean.append(dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
            per_m_low.append( dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
            per_m_high.append(dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy())
        if not per_m_mean:
            return hod_mean_and_band(series, "global")
        mean_ = np.median(np.vstack(per_m_mean), axis=0)
        low   = np.median(np.vstack(per_m_low),  axis=0)
        high  = np.median(np.vstack(per_m_high), axis=0)
        return hrs, mean_, low, high
    raise ValueError("BAND_MODE must be 'global' | 'percentile' | 'within_month'")

if 'scenarios' not in globals():
    raise RuntimeError("Missing `scenarios`. Please run the scenario-building cell first.")

# ---- Precompute stats ----
sys_stats, vil_stats = {}, {}
sys_y, vil_y = [], []

for name, (sys_s, vil_s) in scenarios.items():
    x, s_mean, s_low, s_high = hod_mean_and_band(sys_s, BAND_MODE)
    _, v_mean, v_low, v_high = hod_mean_and_band(vil_s, BAND_MODE)
    sys_stats[name] = (x, s_mean, s_low, s_high)
    vil_stats[name] = (x, v_mean, v_low, v_high)
    sys_y += [s_low.min(), s_high.max(), s_mean.min(), s_mean.max()]
    vil_y += [v_low.min(), v_high.max(), v_mean.min(), v_mean.max()]

sys_ymin, sys_ymax = float(min(sys_y)), float(max(sys_y))
vil_ymin, vil_ymax = float(min(vil_y)), float(max(vil_y))

title_mode = {
    "global": "Global min–max (wide)",
    "percentile": f"P{PCT_LOW}–P{PCT_HIGH} (robust)",
    "within_month": "Within-month median min–max (typical)"
}

# ---- Figure 1: Systems-only (blue, high contrast) ----
fig_sys = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, mean_, low, high = sys_stats[name]

    # Bars (lighter + edge)
    fig_sys.add_bar(x=x, y=mean_,
                    marker=dict(color=SYS_BAR_CLR, line=dict(color=SYS_BAR_EDGE, width=1)),
                    name="Systems mean" if j==1 else None, showlegend=(j==1),
                    row=1, col=j)

    # Band edges (dark) + fill on top for contrast
    fig_sys.add_scatter(x=x, y=high, mode="lines",
                        line=dict(color=SYS_BAND_EDGE, width=1), name=None, showlegend=False, row=1, col=j)
    fig_sys.add_scatter(x=x, y=low, mode="lines", fill="tonexty",
                        fillcolor=SYS_BAND_FILL, line=dict(color=SYS_BAND_EDGE, width=1),
                        name=("Systems band" if j==1 else None), showlegend=(j==1), row=1, col=j)

    fig_sys.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig_sys.update_yaxes(range=[sys_ymin, sys_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_sys.update_layout(
    title=f"Year-averaged daily cycles — Systems only (high-contrast) — band: {title_mode[BAND_MODE]}",
    hovermode="x unified", height=500, barmode="overlay", bargap=BARGAP,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_sys.show()

# ---- Figure 2: Whole-village (yellow, high contrast) ----
fig_vil = make_subplots(rows=1, cols=4, subplot_titles=list(scenarios.keys()), horizontal_spacing=0.04)
for j, name in enumerate(scenarios.keys(), start=1):
    x, mean_, low, high = vil_stats[name]

    # Bars (lighter + edge)
    fig_vil.add_bar(x=x, y=mean_,
                    marker=dict(color=VIL_BAR_CLR, line=dict(color=VIL_BAR_EDGE, width=1)),
                    name="Village mean" if j==1 else None, showlegend=(j==1),
                    row=1, col=j)

    # Band edges (dark) + fill on top for contrast
    fig_vil.add_scatter(x=x, y=high, mode="lines",
                        line=dict(color=VIL_BAND_EDGE, width=1), name=None, showlegend=False, row=1, col=j)
    fig_vil.add_scatter(x=x, y=low, mode="lines", fill="tonexty",
                        fillcolor=VIL_BAND_FILL, line=dict(color=VIL_BAND_EDGE, width=1),
                        name=("Village band" if j==1 else None), showlegend=(j==1), row=1, col=j)

    fig_vil.update_xaxes(title_text="Hour", tickmode="linear", dtick=1, row=1, col=j)
    fig_vil.update_yaxes(range=[vil_ymin, vil_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_vil.update_layout(
    title=f"Year-averaged daily cycles — Whole village (high-contrast) — band: {title_mode[BAND_MODE]}",
    hovermode="x unified", height=500, barmode="overlay", bargap=BARGAP,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_vil.show()


In [9]:
        # Year-averaged daily cycle: TRUE min–max bands
# Village band = Systems band ⊕ sqrt(50)×one-household residual (as defined below).
# Uses existing: systems, household, pd, np, go, TZ

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N = 50  # households

# --- Helpers ---
def _norm_idx(idx):
    i = pd.to_datetime(idx, errors="coerce")
    return i.tz_convert(TZ).tz_localize(None) if getattr(i, "tz", None) is not None else i

def _day_hour_matrix(series: pd.Series) -> pd.DataFrame:
    """Rows = days, Cols = 0..23 hours; sum per hour-of-day to tolerate non-hourly cadence."""
    s = series.dropna()
    return s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)

# --- Systems & Household (align time, build day×hour matrices) ---
sys_ = systems.copy();   sys_.index = _norm_idx(sys_.index)
hh_  = household.copy(); hh_.index  = _norm_idx(hh_.index)

sys_net = (sys_["Produktion"] - sys_["El kWh"]).rename("net_sys").dropna()

hh_imp = next((c for c in hh_.columns if c.lower().strip() in {"el kwh","elkwh"}), None)
hh_prod= next((c for c in hh_.columns if any(k in c.lower() for k in ["produktion","pv","solar","export"])), None)
if hh_imp is None:
    raise KeyError(f"Household import column not found in: {list(hh_.columns)}")
hh_net = ((hh_[hh_prod] if hh_prod else 0) - hh_[hh_imp]).rename("net_household").dropna()

sys_dh = _day_hour_matrix(sys_net)
hh_dh  = _day_hour_matrix(hh_net)

# Align on common set of days
common_days = sys_dh.index.intersection(hh_dh.index)
if common_days.empty:
    raise RuntimeError("No overlapping days between systems and household time series.")
sys_dh = sys_dh.loc[common_days]
hh_dh  = hh_dh.loc[common_days]

hrs = np.arange(24)

# --- Systems stats (mean & TRUE min–max over the whole year) ---
sys_mean = sys_dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
sys_min  = sys_dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
sys_max  = sys_dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()

# --- Village construction using ONE household's residual, year-wide (no monthly splitting) ---
hh_mean_h = hh_dh.mean(axis=0)           # shape: 24
hh_resid  = hh_dh.subtract(hh_mean_h, axis=1)  # residual per (day,hour)
# For each hour h, per-day village = systems_dh[h] + N*hh_mean_h[h] + sqrt(N)*hh_resid[h]
village_cols = []
for h in hrs:
    v_col = sys_dh[h] + (N * hh_mean_h[h]) + (np.sqrt(N) * hh_resid[h])
    village_cols.append(v_col.rename(h))
village_dh = pd.concat(village_cols, axis=1)  # rows=days, cols=0..23

# Village stats (TRUE min–max over whole year)
vil_mean = village_dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
vil_min  = village_dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
vil_max  = village_dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()

# --- Plot: two figures (Systems-only & Whole-village), high-contrast bars + band edges ---
SYS_BAR_CLR   = "rgba(33,150,243,0.25)"; SYS_BAR_EDGE  = "rgb(21,101,192)"
SYS_BAND_FILL = "rgba(33,150,243,0.35)"; SYS_BAND_EDGE = "rgb(13,71,161)"

VIL_BAR_CLR   = "rgba(255,193,7,0.25)";  VIL_BAR_EDGE  = "rgb(204,154,0)"
VIL_BAND_FILL = "rgba(255,193,7,0.40)";  VIL_BAND_EDGE = "rgb(218,145,0)"

# Systems
fig_sys = make_subplots(rows=1, cols=1, subplot_titles=["Systems — Year-averaged daily cycle (TRUE min–max)"])
fig_sys.add_bar(x=hrs, y=sys_mean, marker=dict(color=SYS_BAR_CLR, line=dict(color=SYS_BAR_EDGE, width=1)),
                name="Mean (systems)")
fig_sys.add_scatter(x=hrs, y=sys_max, mode="lines", line=dict(color=SYS_BAND_EDGE, width=1), name=None, showlegend=False)
fig_sys.add_scatter(x=hrs, y=sys_min, mode="lines", fill="tonexty",
                    fillcolor=SYS_BAND_FILL, line=dict(color=SYS_BAND_EDGE, width=1),
                    name="Min–max band (systems)")
fig_sys.update_xaxes(title_text="Hour", tickmode="linear", dtick=1)
fig_sys.update_yaxes(title_text="kWh/h")
fig_sys.update_layout(hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
                      legend=dict(orientation="h", x=0, xanchor="left", y=1.10))
fig_sys.show()

# Village
fig_vil = make_subplots(rows=1, cols=1, subplot_titles=[f"Whole village (N={N}) — Year-averaged daily cycle (TRUE min–max)"])
fig_vil.add_bar(x=hrs, y=vil_mean, marker=dict(color=VIL_BAR_CLR, line=dict(color=VIL_BAR_EDGE, width=1)),
                name="Mean (village)")
fig_vil.add_scatter(x=hrs, y=vil_max, mode="lines", line=dict(color=VIL_BAND_EDGE, width=1), name=None, showlegend=False)
fig_vil.add_scatter(x=hrs, y=vil_min, mode="lines", fill="tonexty",
                    fillcolor=VIL_BAND_FILL, line=dict(color=VIL_BAND_EDGE, width=1),
                    name="Min–max band (village)")
fig_vil.update_xaxes(title_text="Hour", tickmode="linear", dtick=1)
fig_vil.update_yaxes(title_text="kWh/h")
fig_vil.update_layout(hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
                      legend=dict(orientation="h", x=0, xanchor="left", y=1.10))
fig_vil.show()


In [10]:
# Stepwise bands for the already computed arrays:
# assumes you already have: hrs, sys_mean, sys_min, sys_max, vil_mean, vil_min, vil_max
# and the color vars used before (SYS_* and VIL_*). This replaces only the plotting part.

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Systems (stepwise band) ---
fig_sys = make_subplots(rows=1, cols=1, subplot_titles=["Systems — Year-averaged daily cycle (TRUE min–max, step)"])
fig_sys.add_bar(
    x=hrs, y=sys_mean,
    marker=dict(color=SYS_BAR_CLR, line=dict(color=SYS_BAR_EDGE, width=1)),
    name="Mean (systems)"
)
# Max edge (step)
fig_sys.add_scatter(
    x=hrs, y=sys_max, mode="lines",
    line=dict(color=SYS_BAND_EDGE, width=1),
    line_shape="hv",  # ← stepwise band edge
    name=None, showlegend=False
)
# Min edge (step) + fill to previous
fig_sys.add_scatter(
    x=hrs, y=sys_min, mode="lines",
    fill="tonexty", fillcolor=SYS_BAND_FILL,
    line=dict(color=SYS_BAND_EDGE, width=1),
    line_shape="hv",  # ← stepwise band edge
    name="Min–max band (systems)"
)
fig_sys.update_xaxes(title_text="Hour", tickmode="linear", dtick=1)
fig_sys.update_yaxes(title_text="kWh/h")
fig_sys.update_layout(
    hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.10)
)
fig_sys.show()

# --- Whole village (stepwise band) ---
fig_vil = make_subplots(rows=1, cols=1, subplot_titles=[f"Whole village (N=50) — Year-averaged daily cycle (TRUE min–max, step)"])
fig_vil.add_bar(
    x=hrs, y=vil_mean,
    marker=dict(color=VIL_BAR_CLR, line=dict(color=VIL_BAR_EDGE, width=1)),
    name="Mean (village)"
)
# Max edge (step)
fig_vil.add_scatter(
    x=hrs, y=vil_max, mode="lines",
    line=dict(color=VIL_BAND_EDGE, width=1),
    line_shape="hv",
    name=None, showlegend=False
)
# Min edge (step) + fill
fig_vil.add_scatter(
    x=hrs, y=vil_min, mode="lines",
    fill="tonexty", fillcolor=VIL_BAND_FILL,
    line=dict(color=VIL_BAND_EDGE, width=1),
    line_shape="hv",
    name="Min–max band (village)"
)
fig_vil.update_xaxes(title_text="Hour", tickmode="linear", dtick=1)
fig_vil.update_yaxes(title_text="kWh/h")
fig_vil.update_layout(
    hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.10)
)
fig_vil.show()


In [11]:
# Edge-aligned step bands for BOTH figures (reuse your computed arrays):
# hrs: 0..23
# sys_mean, sys_min, sys_max
# vil_mean, vil_min, vil_max
# Color vars: SYS_* and VIL_* from previous cells.

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Helpers to make edge-aligned steps ---
def step_edges_and_y(y24: np.ndarray):
    x_edges = np.arange(-0.5, 24.5, 1)           # 25 edges: -0.5 .. 23.5
    y_step  = np.r_[y24, y24[-1]]                # 25 values: one per interval
    return x_edges, y_step

# ---------- Systems (edge-aligned steps) ----------
x_edges, y_max = step_edges_and_y(sys_max)
_,       y_min = step_edges_and_y(sys_min)

fig_sys = make_subplots(rows=1, cols=1, subplot_titles=["Systems — Year-avg daily (TRUE min–max, edge-aligned)"])

# Bars stay centered at integer hours
fig_sys.add_bar(
    x=np.arange(24), y=sys_mean,
    marker=dict(color=SYS_BAR_CLR, line=dict(color=SYS_BAR_EDGE, width=1)),
    name="Mean (systems)"
)

# Max edge first (step on bin edges)
fig_sys.add_scatter(
    x=x_edges, y=y_max, mode="lines",
    line=dict(color=SYS_BAND_EDGE, width=1),
    line_shape="hv", showlegend=False
)
# Min edge + fill to previous
fig_sys.add_scatter(
    x=x_edges, y=y_min, mode="lines",
    fill="tonexty", fillcolor=SYS_BAND_FILL,
    line=dict(color=SYS_BAND_EDGE, width=1),
    line_shape="hv", name="Min–max band (systems)"
)

fig_sys.update_xaxes(title_text="Hour", range=[-0.5, 23.5], tickmode="linear", dtick=1)
fig_sys.update_yaxes(title_text="kWh/h")
fig_sys.update_layout(
    hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.10)
)
fig_sys.show()

# ---------- Whole village (edge-aligned steps) ----------
x_edges, y_max_v = step_edges_and_y(vil_max)
_,       y_min_v = step_edges_and_y(vil_min)

fig_vil = make_subplots(rows=1, cols=1, subplot_titles=["Whole village — Year-avg daily (TRUE min–max, edge-aligned)"])

fig_vil.add_bar(
    x=np.arange(24), y=vil_mean,
    marker=dict(color=VIL_BAR_CLR, line=dict(color=VIL_BAR_EDGE, width=1)),
    name="Mean (village)"
)

fig_vil.add_scatter(
    x=x_edges, y=y_max_v, mode="lines",
    line=dict(color=VIL_BAND_EDGE, width=1),
    line_shape="hv", showlegend=False
)
fig_vil.add_scatter(
    x=x_edges, y=y_min_v, mode="lines",
    fill="tonexty", fillcolor=VIL_BAND_FILL,
    line=dict(color=VIL_BAND_EDGE, width=1),
    line_shape="hv", name="Min–max band (village)"
)

fig_vil.update_xaxes(title_text="Hour", range=[-0.5, 23.5], tickmode="linear", dtick=1)
fig_vil.update_yaxes(title_text="kWh/h")
fig_vil.update_layout(
    hovermode="x unified", height=420, barmode="overlay", bargap=0.35,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.10)
)
fig_vil.show()


In [12]:
# Year-averaged daily cycle, ALL scenarios → edge-aligned step bands (TRUE min–max) + mean bars
# Uses existing: scenarios = {"Original": (sys_series, vil_series), "Battery-only": (...), "VTX-only": (...), "Battery + VTX": (...)}
# Also uses: pd, np, go already imported in your notebook.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- Guard ----
if 'scenarios' not in globals():
    raise RuntimeError("Missing `scenarios` dict. Build it first (Original / Battery / VTX / Both).")

# ---- Visual style (consistent with prior cells) ----
SYS_BAR_CLR   = "rgba(33,150,243,0.25)"  # blue bars
SYS_BAR_EDGE  = "rgb(21,101,192)"
SYS_BAND_FILL = "rgba(33,150,243,0.35)"
SYS_BAND_EDGE = "rgb(13,71,161)"

VIL_BAR_CLR   = "rgba(255,193,7,0.25)"   # yellow bars
VIL_BAR_EDGE  = "rgb(204,154,0)"
VIL_BAND_FILL = "rgba(255,193,7,0.40)"
VIL_BAND_EDGE = "rgb(218,145,0)"

BARGAP = 0.35
ORDER  = ["Original", "Battery-only", "VTX-only", "Battery + VTX"]  # column order

# ---- Helpers ----
def _day_hour_matrix(series: pd.Series) -> pd.DataFrame:
    s = series.dropna()
    return s.groupby([s.index.normalize(), s.index.hour]).sum().unstack(fill_value=0.0)  # rows=days, cols=0..23

def _hod_mean_minmax(series: pd.Series):
    dh = _day_hour_matrix(series)
    hrs = np.arange(24)
    mean_ = dh.mean(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmin  = dh.min(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    vmax  = dh.max(axis=0).reindex(hrs, fill_value=0.0).to_numpy()
    return hrs, mean_, vmin, vmax

def _edge_step_arrays(y24: np.ndarray):
    x_edges = np.arange(-0.5, 24.5, 1)   # 25 edges: -0.5..23.5
    y_step  = np.r_[y24, y24[-1]]        # 25 values
    return x_edges, y_step

# ---- Precompute per scenario ----
sys_stats, vil_stats = {}, {}
sys_yvals, vil_yvals = [], []

for name in ORDER:
    if name not in scenarios:
        continue
    sys_s, vil_s = scenarios[name]

    # Systems
    x, s_mean, s_min, s_max = _hod_mean_minmax(sys_s)
    xe_s, s_max_step = _edge_step_arrays(s_max)
    _,    s_min_step = _edge_step_arrays(s_min)
    sys_stats[name] = (x, s_mean, xe_s, s_min_step, s_max_step)
    sys_yvals += [s_min.min(), s_max.max(), s_mean.min(), s_mean.max()]

    # Village
    x, v_mean, v_min, v_max = _hod_mean_minmax(vil_s)
    xe_v, v_max_step = _edge_step_arrays(v_max)
    _,    v_min_step = _edge_step_arrays(v_min)
    vil_stats[name] = (x, v_mean, xe_v, v_min_step, v_max_step)
    vil_yvals += [v_min.min(), v_max.max(), v_mean.min(), v_mean.max()]

sys_ymin, sys_ymax = float(min(sys_yvals)), float(max(sys_yvals))
vil_ymin, vil_ymax = float(min(vil_yvals)), float(max(vil_yvals))

# ---- Figure 1: Systems (blue), 1×4, step bands ----
fig_sys = make_subplots(rows=1, cols=len(ORDER), subplot_titles=ORDER, horizontal_spacing=0.04)
for j, name in enumerate(ORDER, start=1):
    if name not in sys_stats: continue
    x, mean_, x_edges, y_min_step, y_max_step = sys_stats[name]

    # mean bars
    fig_sys.add_bar(
        x=x, y=mean_,
        marker=dict(color=SYS_BAR_CLR, line=dict(color=SYS_BAR_EDGE, width=1)),
        name="Systems mean" if j==1 else None, showlegend=(j==1),
        row=1, col=j
    )
    # step band edges + fill
    fig_sys.add_scatter(
        x=x_edges, y=y_max_step, mode="lines",
        line=dict(color=SYS_BAND_EDGE, width=1),
        line_shape="hv", showlegend=False, row=1, col=j
    )
    fig_sys.add_scatter(
        x=x_edges, y=y_min_step, mode="lines",
        fill="tonexty", fillcolor=SYS_BAND_FILL,
        line=dict(color=SYS_BAND_EDGE, width=1),
        line_shape="hv",
        name="Systems min–max" if j==1 else None, showlegend=(j==1),
        row=1, col=j
    )

    fig_sys.update_xaxes(title_text="Hour", range=[-0.5, 23.5], tickmode="linear", dtick=1, row=1, col=j)
    fig_sys.update_yaxes(range=[sys_ymin, sys_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_sys.update_layout(
    title="Year-averaged daily cycles — Systems only (TRUE min–max, edge-aligned steps)",
    hovermode="x unified", height=500, barmode="overlay", bargap=BARGAP,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_sys.show()

# ---- Figure 2: Whole village (yellow), 1×4, step bands ----
fig_vil = make_subplots(rows=1, cols=len(ORDER), subplot_titles=ORDER, horizontal_spacing=0.04)
for j, name in enumerate(ORDER, start=1):
    if name not in vil_stats: continue
    x, mean_, x_edges, y_min_step, y_max_step = vil_stats[name]

    # mean bars
    fig_vil.add_bar(
        x=x, y=mean_,
        marker=dict(color=VIL_BAR_CLR, line=dict(color=VIL_BAR_EDGE, width=1)),
        name="Village mean" if j==1 else None, showlegend=(j==1),
        row=1, col=j
    )
    # step band edges + fill
    fig_vil.add_scatter(
        x=x_edges, y=y_max_step, mode="lines",
        line=dict(color=VIL_BAND_EDGE, width=1),
        line_shape="hv", showlegend=False, row=1, col=j
    )
    fig_vil.add_scatter(
        x=x_edges, y=y_min_step, mode="lines",
        fill="tonexty", fillcolor=VIL_BAND_FILL,
        line=dict(color=VIL_BAND_EDGE, width=1),
        line_shape="hv",
        name="Village min–max" if j==1 else None, showlegend=(j==1),
        row=1, col=j
    )

    fig_vil.update_xaxes(title_text="Hour", range=[-0.5, 23.5], tickmode="linear", dtick=1, row=1, col=j)
    fig_vil.update_yaxes(range=[vil_ymin, vil_ymax], title_text="kWh/h" if j==1 else None, row=1, col=j)

fig_vil.update_layout(
    title="Year-averaged daily cycles — Whole village (TRUE min–max, edge-aligned steps)",
    hovermode="x unified", height=500, barmode="overlay", bargap=BARGAP,
    legend=dict(orientation="h", x=0, xanchor="left", y=1.12)
)
fig_vil.show()
